# trueform: LoRA training and LLM evaluation on a Colab T4

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Allow Google Drive access when asked. Results and the LoRA adapter are copied to `MyDrive/trueform_colab/`.

Rough time on a T4: setup ~5 min, data ~2 min, LoRA training ~1 h, evaluations ~1 h. Nothing here costs money on the free tier.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf trueform && git clone -b main https://github.com/NeuralNerd-hsnpr/trueform
%cd /content/trueform
!git log --oneline -1
!pip install -q -e ".[llm,data,dev]" 

In [ ]:
# Word-safety and metric tests (about 3 minutes)
!pytest -q

Build the data from the pinned sources (hashes are checked into `data/manifest.json`).

In [ ]:
!python scripts/prepare_data.py   # stops if a pinned source file changed upstream
!git status --short data/   # should print nothing: same files as the committed ones

## LoRA fine-tune (Qwen2.5-0.5B-Instruct)

In [ ]:
!python scripts/train_llm.py --config configs/llm-qwen2.5-0.5b-lora.yaml
!cat checkpoints/llm-qwen2.5-0.5b-lora/train_info.json

## Evaluations on the GPU
Fine-tuned + constrained, then the zero-shot baselines.

In [ ]:
for run in ["llm-qwen2.5-0.5b-lora-constrained", "llm-qwen2.5-0.5b-zeroshot-constrained",
            "llm-qwen2.5-0.5b-zeroshot", "llm-qwen2.5-1.5b-zeroshot"]:
    !python scripts/evaluate.py --config configs/eval/{run}.yaml

## Save everything to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/trueform_colab
!cp -r results checkpoints/llm-qwen2.5-0.5b-lora /content/drive/MyDrive/trueform_colab/
!cd /content && zip -qr /content/drive/MyDrive/trueform_colab/trueform_colab_outputs.zip trueform/results trueform/checkpoints/llm-qwen2.5-0.5b-lora
!ls -la /content/drive/MyDrive/trueform_colab/